In [2]:
import pandas as pd
import numpy as np
import os
import time
import pickle
from matplotlib import pyplot as plt
import seaborn as sns

os.chdir(os.getcwd())
os.getcwd()

'/data/gpfs/projects/punim2121/C-Path/outcome_prediction/outcome_prediction'

<hr>

#### DROP MGIT MEASUREMENTS THAT HAVE BEEN PROVED TO BE FALSE POSITIVES. 
#### THIS INFORMATION IS ONLY AVAILABLE IN THE TB-1021 & AND MAYBE TB-1018 STUDY

In [3]:
# load necessary dataframes and study+pat_id data
pat_id_df = pd.read_csv('../data/patients_in_analysis.csv.gz', index_col=0)
#xo=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/xo.csv', low_memory=False)
studies = pat_id_df['STUDYID'].unique()
pat_ids = pat_id_df['USUBJID'].values.tolist()
mb=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/mb_with_results_days_arms.csv', low_memory=False)
#mb=mb[mb['USUBJID'].isin(pat_ids)].dropna(how='all',axis=1)

<hr>

##### **DROP FALSE POSITIVES TTP MEASUREMENTS OF TB-1021 & TB-1018 STUDIES FROM THE MB DATABASE**
* ##### THIS IS MEASURED BY A BLOOD AGAR CULTURE VALIDATION AFTER THE MGIT SIGNALS POSITIVE, 
* ##### IF THE BLOOD-AGAR CULTURE GROWS A NON-MTB BACTERIUM OR FUNGHI, THE TTP WAS FALSE POSITIVE

In [4]:

## Create unique sample reference ID by concatenating Patient ID + Microbiological Reference ID of the sample
mb['SAMPLE_REFID'] = np.nan
mb.loc[~mb['MBREFID'].isna(), 'SAMPLE_REFID'] = mb.loc[~mb['MBREFID'].isna(), 'STUDYID'].astype(str) + \
    '_'+mb.loc[~mb['MBREFID'].isna(), 'MBREFID'].astype(str)
'''
## Extract Sample IDs of false positive samples
false_pos_ttp_ids = mb[(mb['MBTESTCD'] == 'NONMTB') & (
    mb['MBSTRESC'] != 'NEGATIVE (MGIT RESULT VALID)')]['SAMPLE_REFID'].dropna().tolist()


## Drop ALL MEASUREMENTS false positive MGIT measurements
mb_clean = mb.loc[~mb['SAMPLE_REFID'].isin(false_pos_ttp_ids), :]
'''


/tmp/ipykernel_229494/3565020622.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['TB-1021_200335' 'TB-1021_200174' 'TB-1021_200156' ... 'TB-1018_NIX-1930'
 'TB-1018_P1163436' 'TB-1018_DNIX-0350']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  mb.loc[~mb['MBREFID'].isna(), 'SAMPLE_REFID'] = mb.loc[~mb['MBREFID'].isna(), 'STUDYID'].astype(str) + \


"\n## Extract Sample IDs of false positive samples\nfalse_pos_ttp_ids = mb[(mb['MBTESTCD'] == 'NONMTB') & (\n    mb['MBSTRESC'] != 'NEGATIVE (MGIT RESULT VALID)')]['SAMPLE_REFID'].dropna().tolist()\n\n\n## Drop ALL MEASUREMENTS false positive MGIT measurements\nmb_clean = mb.loc[~mb['SAMPLE_REFID'].isin(false_pos_ttp_ids), :]\n"

In [5]:
mb_=mb[mb['USUBJID'].isin(pat_id_df['USUBJID'])]
mb_.groupby(['STUDYID','MBTESTCD','MEDIATYP'],dropna=False).apply(lambda x: (x['MBTSTDTL'].value_counts()))#.reset_index()

/tmp/ipykernel_229494/648449639.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mb_.groupby(['STUDYID','MBTESTCD','MEDIATYP'],dropna=False).apply(lambda x: (x['MBTSTDTL'].value_counts()))#.reset_index()


STUDYID  MBTESTCD  MEDIATYP                  MBTSTDTL                            
TB-1018  AFB       NaN                       Identification                            787
                                             Categorical Count                         130
         MTB       NaN                       Culture Growth                           3629
                                             Time to Detection                         520
         MTBCMPLX  NaN                       MPT64 Antigen Test                        332
                                             Identification                            139
                                             Minimum Cycle Threshold of Detection       10
         NONMTB    BLOOD AGAR                Culture Growth                            631
TB-1020  AFB       NaN                       Categorical Count                       10772
         MTB       LOWENSTEIN JENSEN MEDIUM  Culture Growth                           3705
        

# SUMMARIZE MB TEST RESULT FOR EACH VISIT

## TB-1022

In [6]:
study_name='TB-1022'
mb_subset=mb_[mb_['STUDYID']==study_name].dropna(how='all',axis=1)

def get_mode_of_result(x):
    if len(x['RESULT'].mode())==1:
        return x['RESULT'].mode()[0]
        
    if len(x['RESULT'].mode())==2:
        return 'positive'

    if len(x['RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')   

a=mb_subset.groupby(['USUBJID','MBDY_estimated', 'STUDYID','MBTESTCD','MBMETHOD','MBTSTDTL','MEDIATYP'],dropna=False).apply(get_mode_of_result)

tb_22_std_res=a.reset_index()
tb_22_std_res=tb_22_std_res.rename(columns={0:'STD_RESULT'})
tb_22_std_res['STD_TEST']=np.nan

tb_22_std_res.loc[tb_22_std_res['MBTESTCD']=='AFB','STD_TEST']='ZN-smear'
tb_22_std_res.loc[tb_22_std_res['MBTESTCD']=='MTB','STD_TEST']='LJ-culture'



/tmp/ipykernel_229494/2913694172.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  a=mb_subset.groupby(['USUBJID','MBDY_estimated', 'STUDYID','MBTESTCD','MBMETHOD','MBTSTDTL','MEDIATYP'],dropna=False).apply(get_mode_of_result)
/tmp/ipykernel_229494/2913694172.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ZN-smear' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  tb_22_std_res.loc[tb_22_std_res['MBTESTCD']=='AFB','STD_TEST']='ZN-smear'


## TB-1021

In [18]:
study_name='TB-1021'
mb_subset=mb[mb['STUDYID']==study_name].dropna(how='all',axis=1)
mb_subset[['STD_RESULT']]=mb_subset[['RESULT']].values
print('Num of initial samples',len(mb_subset))

##======== IDENTIFY CONTAMINATED POSITIVE MGIT SAMPLES ===============
#  Based on: https://www.ucl.ac.uk/infection-immunity/sites/infection_immunity/files/remox-laboratory-manual.pdf, page 10-25

## 1. Positive MGIT results are getting tested for contamination by inoculating blood agar with a small sample 
#     from the MGIT tube 
#  2. If the blood agar ('MBTESTCD'=='NONMTB') is negative ('MBSTRESC'=='NEGATIVE (MGIT RESULT VALID)'), 
#     the MGIT is considered not contaminated
not_cont_pos_ttp_sample_ids=mb_subset.loc[(mb_subset['MBTESTCD']=='NONMTB')&\
                                          (mb_subset['MBSTRESC']=='NEGATIVE (MGIT RESULT VALID)'),'SAMPLE_REFID'].dropna().unique()

## 3. Blood agar from contaminated positive MGIT results are NOT REPORTED 
#     ==> we have to infer them by subtracting the not-contaminated MGIT sample IDs from all MGIT sample IDs
all_pos_ttp_sample_ids=mb_subset.loc[(mb_subset['MBTSTDTL']=='Time to Detection')&\
                                     (~mb_subset['MBSTRESN'].isna()),'SAMPLE_REFID'].unique()   

contam_ttp_sample_ids=list(set(all_pos_ttp_sample_ids) - set(not_cont_pos_ttp_sample_ids))
print(len(mb_subset))


##======= DROP CONTAMINATED POSITIVE MGIT SAMPLES + 
#         LINKED ZN-SMEAR RESULTS PERFORMED ON CONT.MGIT + 
#         BLOOD AGAR MEASUREMENTS OF VALIDATION ===============
cont_ttp_idx=mb_subset.loc[(mb_subset['SAMPLE_REFID'].isin(contam_ttp_sample_ids))
                           &(mb_subset['MBGRPID'].str.contains('MGIT'))].index.tolist()

blood_agar_idx=mb_subset.loc[(mb['MBTESTCD']=='NONMTB')].index.tolist()   
mb_subset_clean=mb_subset.drop(index=cont_ttp_idx + blood_agar_idx)
print('Num of samples after dropping contaminated MGIT + blood agar samples',len(mb_subset_clean))



##======== IDENTIFY FALSE POSITIVE MGIT SAMPLES ===============
#  Based on: https://www.ucl.ac.uk/infection-immunity/sites/infection_immunity/files/remox-laboratory-manual.pdf, page 10-25
#            and https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5658986/ , microbiology section
#  1. All cultures that are positive (LJ or MGIT) are getting a ZN-smear from an isolate taken from that culture
#  2. If MGIT is positive but the validation ZN-smear is negative, the MGIT is considered as false positive
#  
## Get samples where the validation ZN-smears of MGIT tests were negative
mgit_samples_with_neg_afb=mb_subset_clean.loc[(mb_subset_clean['MBTESTCD']=='AFB')&\
                      (mb_subset_clean['MBGRPID'].str.contains('MGIT'))&
                      (mb_subset_clean['RESULT']=='negative'),'SAMPLE_REFID'].tolist()

## Get samples where the validation ZN-smears of LJ tests were negative
lj_samples_with_neg_afb=mb_subset_clean.loc[(mb_subset_clean['MBTESTCD']=='AFB')&\
                      (mb_subset_clean['MBGRPID'].str.contains('LJ'))&
                      (mb_subset_clean['RESULT']=='negative'),'SAMPLE_REFID'].tolist()                      

## Extract the results of the matching culture - validation ZN smears
a=(mb_subset_clean[mb_subset_clean['SAMPLE_REFID'].isin(mgit_samples_with_neg_afb)].groupby(['SAMPLE_REFID','MBGRPID','MBTESTCD','MBTSTDTL'],dropna=True).apply(lambda x:x['RESULT'])).reset_index()
a_=(mb_subset_clean[mb_subset_clean['SAMPLE_REFID'].isin(lj_samples_with_neg_afb)].groupby(['SAMPLE_REFID','MBGRPID','MBTESTCD','MBTSTDTL'],dropna=True).apply(lambda x:x['RESULT'])).reset_index()

b=a[a['MBGRPID'].str.contains('MGIT')]
b_=a_[a_['MBGRPID'].str.contains('LJ')]

## Function for checking samples where the MGIT and validating ZN-smear are discordant
def get_false_pos_ttp_samples(x):
    afb_res=x.loc[x['MBTESTCD']=='AFB','RESULT'].values[0]
    try:
        ttp_res=x.loc[x['MBTESTCD']=='MTB','RESULT'].values[0]
        if afb_res=='negative' and ttp_res=='positive':
            return 'false positive'

    ## If IndexError: the MGIT was negative, but a ZN smear was performed nonetheless
    except IndexError:
        pass     


false_pos_mgit_sample_idx=b.groupby(['SAMPLE_REFID','MBGRPID'],dropna=False).apply(get_false_pos_ttp_samples).reset_index()['SAMPLE_REFID'].tolist()
false_pos_lj_sample_idx=b_.groupby(['SAMPLE_REFID','MBGRPID'],dropna=False).apply(get_false_pos_ttp_samples).reset_index()['SAMPLE_REFID'].tolist()
#false_pos_mgit_idx=mb_subset_clean.loc[(mb_subset_clean['SAMPLE_REFID'].isin(false_pos_mgit_sample_idx))&\
#                                        (mb_subset['MBGRPID'].str.contains('MGIT'))].index.tolist()   



##======== SET FALSE POSITIVE MGIT & LJ RESULTS TO NEGATIVE ===============
# MGIT
filt=(mb_subset_clean['SAMPLE_REFID'].isin(false_pos_mgit_sample_idx))&\
                    (mb_subset_clean['MBTSTDTL'].str.contains('Time')).values

mb_subset_clean.loc[filt,['STD_RESULT','STD_NUM_RESULT','STD_CAT_RESULT','STD_NUM_UNITS']]=np.array([['negative',np.nan,0,np.nan]]*filt.sum())

# LJ 
filt=(mb_subset_clean['SAMPLE_REFID'].isin(false_pos_lj_sample_idx))&\
                    (mb_subset_clean['MBLNKID'].str.contains('LJ')).values

mb_subset_clean.loc[filt,['STD_RESULT','STD_NUM_RESULT','STD_CAT_RESULT','STD_NUM_UNITS']]=np.array([['negative',np.nan,0,np.nan]]*filt.sum())      
                           
                    

##======== DROP VALIDATION ZN-SMEARS OF CULTURE ISOLATES ===============
## These ZN-smears serve to check if the culture is false positive. 
#  (true for MGIT, for LJ not sure, as it is not specified in protocol)'
#  Because the information of these samples' positivity is already contained in the LJ or MGIT test result, we can drop these results.                                  

mb_subset_clean=mb_subset_clean.loc[~((mb_subset_clean['MBTESTCD']=='AFB')&\
                                    (mb_subset_clean['MBGRPID'].str.contains('MGIT|LJ'))),:]



##======== MERGE PARALLEL LJ RESULTS REPORTED FOR EACH DAY ===============
## TWO RESULTS OF LJ CULTURE ARE REPORTED FOR EACH SAMPLE: 
#  - AS CULTURE GROWTH (POSITIVE/NEGATIVE) AND  AS CATEGORICAL (NEG/1/2/3/4+)
#  - CALCULATE THE MODE OF THE LJ RESULTS PER DAY. 
#  - A LOW NUMBER OF PATIENTS HAVE 2 SAMPLES TAKEN ON THE SAME DAY ==>TREAT THEM AS PARALLEL SAMPLES AND TAKE THE MODE
#    OVER ALL SAMPLES TAKEN ON A GIVEN DAY
#  - IF THE POSITVE/NEGATIVES ARE TIED, TAKE THE RESULT AS POSITIVE

def get_mode_of_result(x):
    if len(x['STD_RESULT'].mode())==1:
        return x['STD_RESULT'].mode()[0]
        
    if len(x['STD_RESULT'].mode())==2:
        return 'positive'

    if len(x['STD_RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')  

a=(mb_subset_clean.groupby(['USUBJID','MBDY_estimated', 'STUDYID','MBTESTCD','MEDIATYP','MBTSTDTL'],dropna=False,group_keys=True).apply(lambda x: x[['MBMETHOD','STD_NUM_RESULT','STD_CAT_RESULT','STD_NUM_UNITS','STD_RESULT','SAMPLE_REFID']]))
b=a.reset_index()
merged_lj_result=b[~(b['MEDIATYP'].isna()) & (b['MBTESTCD'].str.contains('MTB'))].groupby(['USUBJID','STUDYID','MBDY_estimated','MBTESTCD','MEDIATYP'],dropna=False).apply(get_mode_of_result)
merged_lj_result=merged_lj_result.reset_index()
merged_lj_result=merged_lj_result.rename(columns={0:'STD_RESULT'})
merged_lj_result['MBMETHOD']='MICROBIAL CULTURE, SOLID'                




### MERGE THE CREATED MERGED-LOWENSTEIN DATA WITH THE NON-SOLID CULTURE TEST RESULTS (MGIT, ZN-SMEAR, PCR..)
##  THE ZN-SMEAR IN THIS DATAFRAME ARE NOT VALIDATION ZN-SMEARS!!!
tb_21_std_res=pd.concat([b[b['MEDIATYP'].isna()],merged_lj_result],axis=0,join='outer')
tb_21_std_res=tb_21_std_res.drop(columns=['level_6'])


###=========== STANDARDISE THE NAMES OF AFB, LJ & MGIT CULTURES 
tb_21_std_res['STD_TEST']=np.nan
tb_21_std_res.loc[tb_21_std_res['MBTESTCD']=='AFB','STD_TEST']='ZN-smear'
tb_21_std_res.loc[tb_21_std_res['MBTESTCD']=='MTBCMPLX','STD_TEST']='AccuProbe'

tb_21_std_res.loc[(tb_21_std_res['MBTESTCD']=='MTB')\
                  &(tb_21_std_res['MBMETHOD'].str.contains('SOLID')),'STD_TEST']='LJ-culture'
tb_21_std_res.loc[(tb_21_std_res['MBTESTCD']=='MTB')\
                  &(tb_21_std_res['MBMETHOD'].str.contains('LIQUID')),'STD_TEST']='MGIT'                  



Num of initial samples 176592
176592
Num of samples after dropping contaminated MGIT + blood agar samples 154066


/tmp/ipykernel_229494/3145512701.py:54: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  a=(mb_subset_clean[mb_subset_clean['SAMPLE_REFID'].isin(mgit_samples_with_neg_afb)].groupby(['SAMPLE_REFID','MBGRPID','MBTESTCD','MBTSTDTL'],dropna=True).apply(lambda x:x['RESULT'])).reset_index()
/tmp/ipykernel_229494/3145512701.py:55: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  a_=(mb_subset_clean[mb_subset_clean['SAMP

In [20]:
tb_21_std_res

,USUBJID,MBDY_estimated,STUDYID,MBTESTCD,MEDIATYP,MBTSTDTL,MBMETHOD,STD_NUM_RESULT,STD_CAT_RESULT,STD_NUM_UNITS,STD_RESULT,SAMPLE_REFID,STD_TEST
0,TB-1021/1001485,-2.0,TB-1021,AFB,NaN,Categorical Count,ZIEHL NEELSEN ACID FAST STAIN,NaN,2+,NaN,positive,TB-1021_300001,ZN-smear
3,TB-1021/1001485,-2.0,TB-1021,MTB,NaN,Time to Detection,"MICROBIAL CULTURE, LIQUID",6.125,6.125,DAYS,positive,TB-1021_300001,MGIT
4,TB-1021/1001485,1.0,TB-1021,AFB,NaN,Categorical Count,ZIEHL NEELSEN ACID FAST STAIN,NaN,4+,NaN,positive,TB-1021_300002,ZN-smear
7,TB-1021/1001485,1.0,TB-1021,MTB,NaN,Time to Detection,"MICROBIAL CULTURE, LIQUID",4.417,4.417,DAYS,positive,TB-1021_300002,MGIT
8,TB-1021/1001485,1.0,TB-1021,MTBCMPLX,NaN,Identification,NUCLEIC ACID HYBRIDIZATION,NaN,PRESENT,NaN,positive,TB-1021_300002,AccuProbe
...,...,...,...,...,...,...,...,...,...,...,...,...,...
31562,TB-1021/3062226,183.0,TB-1021,MTB,LOWENSTEIN JENSEN MEDIUM,NaN,"MICROBIAL CULTURE, SOLID",NaN,NaN,NaN,positive,NaN,LJ-culture
31563,TB-1021/3062226,274.0,TB-1021,MTB,LOWENSTEIN JENSEN MEDIUM,NaN,"MICROBIAL CULTURE, SOLID",NaN,NaN,NaN,positive,NaN,LJ-culture
31564,TB-1021/3062226,366.0,TB-1021,MTB,LOWENSTEIN JENSEN MEDIUM,NaN,"MICROBIAL CULTURE, SOLID",NaN,NaN,NaN,negative,NaN,LJ-culture
31565,TB-1021/3062226,456.0,TB-1021,MTB,LOWENSTEIN JENSEN MEDIUM,NaN,"MICROBIAL CULTURE, SOLID",NaN,NaN,NaN,negative,NaN,LJ-culture


### Save patient ID of contaminated & false positive samples for TB-1021

In [12]:
contam_ttp_sample_ids = [x.split('TB-1021_')[-1] for x in contam_ttp_sample_ids]
false_pos_mgit_sample_idx = [x.split('TB-1021_')[-1] for x in false_pos_mgit_sample_idx]


fn='../data/tb_1021_contam_ttp_sample_ids.pickle'
with open(fn, 'wb') as handle:
    pickle.dump(contam_ttp_sample_ids, handle)


fn='../data/tb_1021_false_pos_mgit_sample_idx.pickle'
with open(fn, 'wb') as handle:
    pickle.dump(false_pos_mgit_sample_idx, handle)

# TB-1020

In [11]:
study_name='TB-1020'
mb_subset=mb[mb['STUDYID']==study_name].dropna(how='all',axis=1)
mb_subset[['STD_RESULT']]=mb_subset[['RESULT']].values
print('Num of initial samples',len(mb_subset))

def get_mode_of_result(x):
    if len(x['RESULT'].mode())==1:
        return x['RESULT'].mode()[0]
        
    if len(x['RESULT'].mode())==2:
        return 'positive'

    if len(x['RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")') 


## MGIT results are reported as categorical values, but for the baseline and first follow-up visit, 
#  the TTP values are also recorded 
#   ==> frop the TTP values, as the categorical results are anyway recorded 
dupl_mgit=mb_subset[mb_subset['SPDEVID'].str.contains('MGIT')\
            &(mb_subset.duplicated(subset=['SAMPLE_REFID','MBDY','SPDEVID','MBTESTCD'],keep=False))]

dupl_mgit_idx=dupl_mgit[dupl_mgit['MBTSTDTL'].str.contains('Time')].index.tolist()

mb_subset_clean=mb_subset.drop(index=dupl_mgit_idx)


## Get mode of multiple measurements with same method from same sample 
#  ==> if tie between positive and negative, consider as positive
c=mb_subset_clean.groupby(['USUBJID','MBDY_estimated', 'STUDYID','SPDEVID','SAMPLE_REFID','MBTESTCD','MBMETHOD','MEDIATYP',],dropna=False).apply(get_mode_of_result).to_frame()
c.columns=['STD_RESULT']



  

###=========== STANDARDISE THE NAMES OF AFB, LJ & MGIT CULTURES 
tb_20_std_res=c.reset_index()

tb_20_std_res['STD_TEST']=np.nan
tb_20_std_res.loc[(tb_20_std_res['MBTESTCD']=='AFB')\
                  &(tb_20_std_res['MBMETHOD'].str.contains('ZIEHL')),'STD_TEST']='ZN-smear'

tb_20_std_res.loc[(tb_20_std_res['MBTESTCD']=='AFB')\
                  &(tb_20_std_res['MBMETHOD'].str.contains('AURAMINE')),'STD_TEST']='Auramine-smear'

tb_20_std_res.loc[(tb_20_std_res['MBTESTCD']=='MTBCMPLX')\
                  &(tb_20_std_res['MBMETHOD'].isna()),'STD_TEST']='MTB-complex'

tb_20_std_res.loc[(tb_20_std_res['MBTESTCD']=='MTB')\
                  &(tb_20_std_res['MBMETHOD'].str.contains('NUCLEIC')),'STD_TEST']='HAIN-test'

tb_20_std_res.loc[(tb_20_std_res['MBTESTCD']=='MTB')\
                  &(tb_20_std_res['MBMETHOD'].str.contains('SOLID')),'STD_TEST']='LJ-culture'
tb_20_std_res.loc[(tb_20_std_res['MBTESTCD']=='MTB')\
                  &(tb_20_std_res['MBMETHOD'].str.contains('LIQUID')),'STD_TEST']='MGIT'    
                

Num of initial samples 25169


/tmp/ipykernel_229494/2585229351.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  c=mb_subset_clean.groupby(['USUBJID','MBDY_estimated', 'STUDYID','SPDEVID','SAMPLE_REFID','MBTESTCD','MBMETHOD','MEDIATYP',],dropna=False).apply(get_mode_of_result).to_frame()
/tmp/ipykernel_229494/2585229351.py:41: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ZN-smear' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  tb_20_std_res.loc[(tb_20_std_res['MBTESTCD']=='AFB')\


In [16]:
tb_20_std_res

,USUBJID,MBDY_estimated,STUDYID,SPDEVID,SAMPLE_REFID,MBTESTCD,MBMETHOD,MEDIATYP,STD_RESULT,STD_TEST
0,TB-1020/1001,1.0,TB-1020,HAIN,TB-1020_MB-1001-020-0401,MTB,NUCLEIC ACID HYBRIDIZATION,NaN,positive,HAIN-test
1,TB-1020/1001,1.0,TB-1020,MGIT,TB-1020_MB-1001-020-0401,MTB,"MICROBIAL CULTURE, LIQUID",NaN,positive,MGIT
2,TB-1020/1001,1.0,TB-1020,NaN,TB-1020_MB-1001-020-0401,AFB,ZIEHL NEELSEN ACID FAST STAIN,NaN,positive,ZN-smear
3,TB-1020/1001,1.0,TB-1020,NaN,TB-1020_MB-1001-020-0401,MTBCMPLX,NaN,NaN,positive,MTB-complex
4,TB-1020/1001,87.0,TB-1020,MGIT,TB-1020_MB-1001-050-0487,MTB,"MICROBIAL CULTURE, LIQUID",NaN,negative,MGIT
...,...,...,...,...,...,...,...,...,...,...
21497,TB-1020/6089,353.0,TB-1020,NaN,TB-1020_MB-6089-140-0753,MTB,"MICROBIAL CULTURE, SOLID",LOWENSTEIN JENSEN MEDIUM,negative,LJ-culture
21498,TB-1020/6089,437.0,TB-1020,NaN,TB-1020_MB-6089-150-0837,AFB,AURAMINE STAIN,NaN,negative,Auramine-smear
21499,TB-1020/6089,437.0,TB-1020,NaN,TB-1020_MB-6089-150-0837,MTB,"MICROBIAL CULTURE, SOLID",LOWENSTEIN JENSEN MEDIUM,negative,LJ-culture
21500,TB-1020/6089,521.0,TB-1020,NaN,TB-1020_MB-6089-160-0921,AFB,AURAMINE STAIN,NaN,negative,Auramine-smear


# TB-1018

In [14]:
study_name='TB-1018'
mb_subset=mb[mb['STUDYID']==study_name].dropna(how='all',axis=1)
mb_subset[['STD_RESULT']]=mb_subset[['RESULT']].values
print('Num of initial samples',len(mb_subset))

mb_subset.groupby(['STUDYID','MBTESTCD','MEDIATYP'],dropna=False).apply(lambda x: (x['MBTSTDTL'].value_counts()))#.reset_index()


##======== IDENTIFY CONTAMINATED POSITIVE MGIT SAMPLES ===============
#  Based on: https://www.nejm.org/doi/suppl/10.1056/NEJMoa1901814/suppl_file/nejmoa1901814_protocol.pdf page 410-455
#  MGIT :450-455
#  AFB: 433-436

## IMPORTANT: 
#  -POSITIVE MGIT RESULTS ARE REPORTED AS: TTP + CULTURE GROWTH
#  -NEGATIVE MGIT RESULTS ARE REPORTED AS: ONLY CULTURE GROWTH

## 1. Positive MGIT results are getting tested for contamination by inoculating blood agar with a small sample 
#     from the MGIT tube 
#  2. If the blood agar ('MBTESTCD'=='NONMTB') is not negative ('MBSTRESC'!='NEGATIVE (MGIT RESULT VALID)'), 
#     the MGIT is considered contaminated
cont_pos_ttp_sample_ids=mb_subset.loc[(mb_subset['MBTESTCD']=='NONMTB')\
                                         &(mb_subset['MBSTRESC']!='NEGATIVE (MGIT RESULT VALID)')\
                                        ,'SAMPLE_REFID'].dropna().unique().tolist()

## 3. Check for samples, where Blood Agar culturing was not done ==> based on protocol, these should be considered as invalid 
#     ==> we have to infer them by subtracting the not-contaminated MGIT sample IDs from all MGIT sample IDs
all_pos_ttp_sample_ids=mb_subset.loc[(mb_subset['SPDEVID'].str.contains('MGIT',na=False))\
                                    &(mb_subset['MBMETHOD'].str.contains('MICROBIAL CULTURE, LIQUID',na=False))\
                                    &(mb_subset['MBSTRESC']!='NEGATIVE')\
                                     ,'SAMPLE_REFID'].unique() 

ttp_with_blood_agar_sample_ids=mb_subset.loc[(mb_subset['MBTESTCD']=='NONMTB')\
                                            #&(mb_subset['MBSTRESC']=='NEGATIVE (MGIT RESULT VALID)')\
                                            ,'SAMPLE_REFID'].dropna().unique()                                       

## Get samplesIds where no blood agar is available
pos_ttp_wo_blood_agar_sample_ids=list(set(all_pos_ttp_sample_ids) - set(ttp_with_blood_agar_sample_ids))

## Invalid MGIT sample ids = contaminated samples + samples without blood agar measurement
invalid_ttp_sample_ids = cont_pos_ttp_sample_ids + pos_ttp_wo_blood_agar_sample_ids

print(len(mb_subset))



##======= DROP CONTAMINATED POSITIVE MGIT SAMPLES + 
#         LINKED ZN-SMEAR RESULTS PERFORMED ON CONT.MGIT + 
#         BLOOD AGAR MEASUREMENTS OF VALIDATION ===============
cont_ttp_idx=mb_subset.loc[(mb_subset['SAMPLE_REFID'].isin(cont_pos_ttp_sample_ids))
                           &(mb_subset['SPDEVID'].str.contains('MGIT'))].index.tolist()

inv_ttp_idx=mb_subset.loc[(mb_subset['SAMPLE_REFID'].isin(invalid_ttp_sample_ids))\
                            &(mb_subset['SPDEVID'].str.contains('MGIT',na=False))\
                            &(mb_subset['MBMETHOD'].str.contains('MICROBIAL CULTURE, LIQUID',na=False))].index.tolist()

inv_afb_idx=mb_subset.loc[(mb_subset['SAMPLE_REFID'].isin(invalid_ttp_sample_ids))\
                            &(mb_subset['MBTESTCD'].str.contains('AFB',na=False))].index.tolist()

blood_agar_idx=mb_subset.loc[(mb['MBTESTCD']=='NONMTB')].index.tolist()   
mb_subset_clean=mb_subset.drop(index=inv_ttp_idx + blood_agar_idx +inv_afb_idx)
print('Num of samples after dropping contaminated MGIT + blood agar samples',len(mb_subset_clean))


##======== IDENTIFY CONTAMINATED/ MGIT SAMPLES, BASED ON COMMENTS OF THE MICROBIOLOGY LAB ===============
## List containing positive samples' comments, which describe some issues with the sample, but don't mean they are contaminated
not_cont_comments=['ZN REPEATED AND STILL NO AFB SEEN, BUT THE ID TEST CONFIRMED PRESENCE OF MTB COMPLEX.',
                    'MGIT TUBE NOT REGISTERED ON EPICENTRE IN ERROR, THEREFORE NO RESULT TIME',
                    'RESULT TIME NOT AVAILABLE DUE TO PROBLEM WITH EPICENTRE DATABASE',
                    'RESULT TIME NOT AVAILABLE DUE TO EPICENTER',
                    'RESULT TIME NOT AVAILABLE DUE TO EPICENTER ERROR.',
                    'RESULT TIME NOT AVAILABLE DUE TO EPICENTER ERROR',
                    'ZN WAS REPEATED AND STILL NO AFB SEEN, BUT ID TEST CONFIRMED PRESENCE OF MTB COMPLEX. ZN WAS REPEATED AGAIN AFTER REPORTING AND ZN AFB PRESENT',
                    'FALSE POSITIVE']

## Take samples, that are positive, their comment is not in the list & is not NaN (which means the results is valid)
cont_sample_sample_ids=mb_subset_clean.loc[(~mb_subset_clean['COMMENT'].isin(not_cont_comments))\
                                    &(~mb_subset_clean['COMMENT'].isna())\
                                    &(mb_subset_clean['RESULT']=='positive')\
                                    ,'SAMPLE_REFID'].unique().tolist()


mb_subset_clean=mb_subset_clean[~mb_subset_clean['SAMPLE_REFID'].isin(cont_sample_sample_ids)]


##======== IDENTIFY FALSE POSITIVE MGIT SAMPLES ===============
# Based on: https://www.nejm.org/doi/suppl/10.1056/NEJMoa1901814/suppl_file/nejmoa1901814_protocol.pdf page 410-455
# AFB:  - performed as auramine staining at baseline for MTB detection (keep these)
#       - ZN staining for validation of MGIT results
#  1. All cultures that are positive (LJ or MGIT) are getting a ZN-smear from an isolate taken from that culture
#  2. If MGIT is positive but the validation ZN-smear is negative, the MGIT is considered as false positive
#  
## Get samples where the validation ZN-smears of MGIT tests were negative


mgit_samples_with_neg_afb=mb_subset_clean.loc[(mb_subset_clean['MBTESTCD']=='AFB')&\
                      (~mb_subset_clean['MBSPID'].isna())& # smear belongs to sample from MGIT tube
                      (mb_subset_clean['RESULT']=='negative'),'SAMPLE_REFID'].unique().tolist()
                

## Extract the results of the matching culture - validation ZN smears
a=(mb_subset_clean[mb_subset_clean['SAMPLE_REFID'].isin(mgit_samples_with_neg_afb)].groupby(['SAMPLE_REFID','SPDEVID','MBGRPID','MBTESTCD','MBTSTDTL'],dropna=False).apply(lambda x:x['RESULT'])).reset_index()


## Function for checking samples where the MGIT and validating ZN-smear are discordant
def get_false_pos_ttp_samples(x):
    try:
        afb_res=x.loc[(x['MBTESTCD']=='AFB')\
                    &(x['MBTSTDTL']=='Identification'),'RESULT'].values[0]
        try:
            ttp_res=x.loc[x['MBTESTCD']=='MTB','RESULT'].values[0]
            
            if len(x.loc[x['MBTESTCD']=='MTB'])==0:
                print(x['SAMPLE_REFID'].unique())
                pass
            else:
                if afb_res=='negative' and ttp_res=='positive':
                    return 'false positive'

        ## If IndexError: the MGIT was negative, but a ZN smear was performed nonetheless
        except IndexError:
            pass     
    except IndexError:
            pass     

## Get possible false positives (manual page 454) and drop them, as these samples 
#  (defined by MBGRPID, as multiple samples can be taken from a sputum) have a negative AFB with a positive MGIT
false_pos_mgit_sample_idx=a.groupby(['SAMPLE_REFID','MBGRPID'],dropna=False).apply(get_false_pos_ttp_samples).reset_index()['MBGRPID'].tolist()
  
mb_subset_clean=mb_subset_clean[~mb_subset_clean['MBGRPID'].isin(false_pos_mgit_sample_idx)]

##======== DROP VALIDATION ZN-SMEARS OF CULTURE ISOLATES ===============
## These ZN-smears serve to check if the culture is false positive. 
#  (true for MGIT, for LJ not sure, as it is not specified in protocol)'
#  Because the information of these samples' positivity is already contained in the LJ or MGIT test result, we can drop these results.                                  

mb_subset_clean=mb_subset_clean.loc[~((mb_subset_clean['MBTESTCD']=='AFB')&\
                                    (mb_subset_clean['MBTSTDTL']=='Identification')),:]




##======== MERGE PARALLEL LJ RESULTS REPORTED FOR EACH DAY ===============
## TWO RESULTS OF LJ CULTURE ARE REPORTED FOR EACH SAMPLE: 
#  - AS CULTURE GROWTH (POSITIVE/NEGATIVE) AND  AS CATEGORICAL (NEG/1/2/3/4+)
#  - CALCULATE THE MODE OF THE LJ RESULTS PER DAY. 
#  - A LOW NUMBER OF PATIENTS HAVE 2 SAMPLES TAKEN ON THE SAME DAY ==>TREAT THEM AS PARALLEL SAMPLES AND TAKE THE MODE
#    OVER ALL SAMPLES TAKEN ON A GIVEN DAY
#  - IF THE POSITVE/NEGATIVES ARE TIED, TAKE THE RESULT AS POSITIVE

def get_mode_of_result(x):
    if len(x['STD_RESULT'].mode())==1:
        return x['STD_RESULT'].mode()[0]
        
    if len(x['STD_RESULT'].mode())==2:
        return 'positive'

    if len(x['STD_RESULT'].mode())>2:
        raise ValueError('Invalid mb result! (not "negative" or "positive")')  

a=(mb_subset_clean.groupby(['USUBJID','MBDY_estimated', 'STUDYID','MBTESTCD','MEDIATYP','MBTSTDTL','MBMETHOD'],dropna=False,group_keys=True).apply(lambda x: x[['STD_NUM_RESULT','STD_CAT_RESULT','STD_NUM_UNITS','STD_RESULT','SAMPLE_REFID']]))
b=a.reset_index()
merged_result=b.groupby(['USUBJID','MBDY_estimated','STUDYID','MBTESTCD','MBTSTDTL','MBMETHOD'],dropna=False).apply(get_mode_of_result)
merged_result=merged_result.reset_index()
merged_result=merged_result.rename(columns={0:'STD_RESULT'})
#merged_result['MBMETHOD']=np.nan
#merged_result.loc[merged_result['MBTESTCD']=='MTB','MBMETHOD']='MICROBIAL CULTURE, LIQUID'    



###=========== STANDARDISE THE NAMES OF AFB, LJ & MGIT CULTURES 
tb_18_std_res=merged_result.copy()

tb_18_std_res['STD_TEST']=np.nan
tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='AFB')\
                  &(tb_18_std_res['MBMETHOD'].str.contains('ACID FAST')),'STD_TEST']='ZN-smear'

tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='AFB')\
                  &(tb_18_std_res['MBMETHOD'].str.contains('AURAMINE')),'STD_TEST']='Auramine-smear'


tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='MTBCMPLX')\
                  &(tb_18_std_res['MBMETHOD'].isna()),'STD_TEST']='MTB-complex'

tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='MTBCMPLX')\
                  &(tb_18_std_res['MBMETHOD'].str.contains('LINE')),'STD_TEST']='HAIN-test'

tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='MTBCMPLX')\
                  &(tb_18_std_res['MBMETHOD'].str.contains('IMMUNOCHROMATOGRAPHY')),'STD_TEST']='MPT64-Antigen-Test'

tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='MTBCMPLX')\
                  &(tb_18_std_res['MBMETHOD'].str.contains('POLYMERASE')),'STD_TEST']='RT-PCR'


tb_18_std_res.loc[(tb_18_std_res['MBTESTCD']=='MTB')\
                  &(tb_18_std_res['MBMETHOD'].str.contains('LIQUID')),'STD_TEST']='MGIT'    

Num of initial samples 6729
6729
Num of samples after dropping contaminated MGIT + blood agar samples 5644


/tmp/ipykernel_229494/4272176378.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mb_subset.groupby(['STUDYID','MBTESTCD','MEDIATYP'],dropna=False).apply(lambda x: (x['MBTSTDTL'].value_counts()))#.reset_index()
/tmp/ipykernel_229494/4272176378.py:102: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  a=(mb_subset_clean[mb_subset_clean['SAMPLE_REFID'].isin(mgit_samples_with_neg_afb)].groupby(['SAMPLE_REFID','

In [40]:
merged_result['MBMETHOD'].value_counts(dropna=False)

MICROBIAL CULTURE, LIQUID                    2314
IMMUNOCHROMATOGRAPHY                          218
LINE PROBE ASSAY                               71
AURAMINE STAIN                                 64
NaN                                            60
ACID FAST STAIN                                53
REAL-TIME POLYMERASE CHAIN REACTION ASSAY      13
Name: MBMETHOD, dtype: int64

In [15]:
tb_18_std_res#['STD_TEST'].value_counts(dropna=False)

,USUBJID,MBDY_estimated,STUDYID,MBTESTCD,MBTSTDTL,MBMETHOD,STD_RESULT,STD_TEST
0,TB-1018/01-9002,-9.0,TB-1018,AFB,Categorical Count,ACID FAST STAIN,negative,ZN-smear
1,TB-1018/01-9002,1.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
2,TB-1018/01-9002,7.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
3,TB-1018/01-9002,14.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
4,TB-1018/01-9002,28.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",positive,MGIT
...,...,...,...,...,...,...,...,...
2788,TB-1018/04-9014,218.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
2789,TB-1018/04-9014,279.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
2790,TB-1018/04-9014,352.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT
2791,TB-1018/04-9014,453.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT


# CONCATENATE STANDARDISED DATAFRAMES OF ALL STUDIES

In [21]:
mb_std_res=pd.concat([tb_18_std_res,tb_20_std_res,tb_21_std_res,tb_22_std_res],axis=0)
mb_std_res=mb_std_res.rename(columns={'STD_TEST':'STD_MBTEST'})
mb_std_res['USUBJID'] = mb_std_res['USUBJID'].str.replace('\\', '/', regex=False)
mb_std_res.to_csv('../data/out_mb_wo_false_positives.csv.gz',compression='gzip')

In [ ]:
mb_std_res#['STD_MBTEST'].value_counts(dropna=False)

In [22]:
mb_std_res

,USUBJID,MBDY_estimated,STUDYID,MBTESTCD,MBTSTDTL,MBMETHOD,STD_RESULT,STD_MBTEST,SPDEVID,SAMPLE_REFID,MEDIATYP,STD_NUM_RESULT,STD_CAT_RESULT,STD_NUM_UNITS
0,TB-1018/01-9002,-9.0,TB-1018,AFB,Categorical Count,ACID FAST STAIN,negative,ZN-smear,NaN,NaN,NaN,NaN,NaN,NaN
1,TB-1018/01-9002,1.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT,NaN,NaN,NaN,NaN,NaN,NaN
2,TB-1018/01-9002,7.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT,NaN,NaN,NaN,NaN,NaN,NaN
3,TB-1018/01-9002,14.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",negative,MGIT,NaN,NaN,NaN,NaN,NaN,NaN
4,TB-1018/01-9002,28.0,TB-1018,MTB,Culture Growth,"MICROBIAL CULTURE, LIQUID",positive,MGIT,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41151,TB-1022/53506,114.0,TB-1022,MTB,Culture Growth,"MICROBIAL CULTURE, SOLID",negative,LJ-culture,NaN,NaN,LOWENSTEIN JENSEN MEDIUM,NaN,NaN,NaN
41152,TB-1022/53506,200.0,TB-1022,AFB,Categorical Count,NaN,positive,ZN-smear,NaN,NaN,NaN,NaN,NaN,NaN
41153,TB-1022/53506,201.0,TB-1022,AFB,Categorical Count,NaN,positive,ZN-smear,NaN,NaN,NaN,NaN,NaN,NaN
41154,TB-1022/53506,201.0,TB-1022,MTB,Culture Growth,"MICROBIAL CULTURE, SOLID",positive,LJ-culture,NaN,NaN,LOWENSTEIN JENSEN MEDIUM,NaN,NaN,NaN
